# 04 - Mineração de Padrões

Detecção de regimes ocultos e padrões exploráveis:
1. Hidden Markov Models (HMM)
2. Padrões temporais (hora/dia)
3. Concept Drift
4. Sequências recorrentes

In [ ]:
import sys
sys.path.insert(0, '.')
from config_analysis import *

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import plotly.io as pio
pio.templates.default = 'plotly_dark'

CYAN, MAGENTA, GREEN, RED, YELLOW = '#00f0ff', '#ff00ff', '#00ff88', '#ff3366', '#ffff00'

df = load_processed_data()
if df is None:
    raise FileNotFoundError("Execute o notebook 01 primeiro!")

print(f"Dataset: {len(df):,} registros")

## 1. Hidden Markov Models (HMM)

Detectar estados ocultos (regimes) que o jogo pode ter.

In [ ]:
try:
    from hmmlearn.hmm import GaussianHMM
    
    # Usar amostra para treino (HMM é lento em 3.9M)
    sample_size = 100_000
    df_sample = df.tail(sample_size).copy()
    
    # Features para HMM
    features = df_sample[['multiplicador', 'rolling_std_20', 'pct_low_20']].dropna().values
    
    # Testar 2, 3 e 4 estados
    results = {}
    for n_states in [2, 3, 4]:
        model = GaussianHMM(
            n_components=n_states,
            covariance_type='full',
            n_iter=100,
            random_state=42
        )
        model.fit(features)
        score = model.score(features)
        states = model.predict(features)
        results[n_states] = {
            'model': model,
            'score': score,
            'bic': -2 * score + n_states * np.log(len(features)),
            'states': states,
        }
        print(f"  {n_states} estados: log-likelihood = {score:.0f}, BIC = {results[n_states]['bic']:.0f}")
    
    # Selecionar melhor modelo (menor BIC)
    best_n = min(results, key=lambda k: results[k]['bic'])
    print(f"\nMelhor modelo: {best_n} estados (BIC = {results[best_n]['bic']:.0f})")
    
    best = results[best_n]
    states = best['states']
    
except ImportError:
    print("Instale hmmlearn: pip install hmmlearn")
    print("Depois execute esta célula novamente.")
    best_n = 2
    states = None

In [ ]:
if states is not None:
    df_sample_valid = df_sample.iloc[-len(states):].copy()
    df_sample_valid['state'] = states
    
    # Estatísticas por estado
    print("\nEstatísticas por estado HMM:")
    print("=" * 60)
    for state in range(best_n):
        mask = df_sample_valid['state'] == state
        sub = df_sample_valid[mask]
        print(f"\nEstado {state} ({mask.sum():,} registros, {mask.mean()*100:.1f}%):")
        print(f"  Multiplicador médio:  {sub['multiplicador'].mean():.3f}x")
        print(f"  Multiplicador mediana: {sub['multiplicador'].median():.3f}x")
        print(f"  % LOW:                {sub['is_low'].mean()*100:.1f}%")
        print(f"  Volatilidade (std):   {sub['multiplicador'].std():.3f}")
        print(f"  Max streak LOW:       {sub['low_streak'].max()}")
    
    # Visualizar estados ao longo do tempo
    colors = [CYAN, MAGENTA, GREEN, YELLOW]
    fig = go.Figure()
    for state in range(best_n):
        mask = df_sample_valid['state'] == state
        fig.add_trace(go.Scatter(
            x=df_sample_valid.loc[mask, 'date'],
            y=df_sample_valid.loc[mask, 'multiplicador'].clip(upper=20),
            mode='markers', name=f'Estado {state}',
            marker=dict(color=colors[state], size=2, opacity=0.5),
        ))
    
    fig.update_layout(
        title=f'Regimes HMM ({best_n} estados) - Últimos {sample_size:,} rounds',
        xaxis_title='', yaxis_title='Multiplicador',
        height=500
    )
    fig.show()

## 2. Padrões Temporais (Hora x Dia)

In [ ]:
# Win rate da estratégia por hora x dia
trigger = STRATEGY_TRIGGER
target = STRATEGY_TARGET

signals = df[df['low_streak'] == trigger].copy()
valid_idx = signals.index[signals.index < len(df) - 1]
signals_valid = signals.loc[valid_idx].copy()
signals_valid['hit'] = (df.loc[valid_idx + 1, 'multiplicador'].values >= target).astype(int)

# Heatmap de win rate por hora x dia
wr_heatmap = signals_valid.groupby(['dia_semana', 'hora'])['hit'].mean().unstack(fill_value=0) * 100
dias = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']

fig = go.Figure(go.Heatmap(
    z=wr_heatmap.values,
    x=[str(h) for h in wr_heatmap.columns],
    y=[dias[i] for i in wr_heatmap.index],
    colorscale='RdYlGn',
    colorbar=dict(title='Win Rate %'),
    text=np.round(wr_heatmap.values, 1),
    texttemplate='%{text}%',
    textfont=dict(size=9),
))

fig.update_layout(
    title=f'Win Rate (%) da Estratégia por Hora x Dia (trigger={trigger}, target={target}x)',
    xaxis_title='Hora', yaxis_title='',
    height=400
)
fig.show()

# Melhores e piores horários
wr_by_hour = signals_valid.groupby('hora').agg(
    win_rate=('hit', 'mean'),
    sinais=('hit', 'count')
).reset_index()
wr_by_hour['win_rate'] *= 100
wr_by_hour = wr_by_hour[wr_by_hour['sinais'] >= 10]  # mínimo de sinais

print("\nTop 5 melhores horários:")
best = wr_by_hour.nlargest(5, 'win_rate')
for _, row in best.iterrows():
    print(f"  {int(row['hora']):02d}h: {row['win_rate']:.1f}% ({int(row['sinais'])} sinais)")

print("\nTop 5 piores horários:")
worst = wr_by_hour.nsmallest(5, 'win_rate')
for _, row in worst.iterrows():
    print(f"  {int(row['hora']):02d}h: {row['win_rate']:.1f}% ({int(row['sinais'])} sinais)")

## 3. Concept Drift - Como os Padrões Mudam

In [ ]:
# Win rate da estratégia por mês (concept drift)
signals_valid['ano_mes'] = signals_valid['date'].dt.strftime('%Y-%m')

monthly_wr = signals_valid.groupby('ano_mes').agg(
    win_rate=('hit', 'mean'),
    sinais=('hit', 'count')
).reset_index()
monthly_wr['win_rate'] *= 100

# Rolling win rate (janela de 3 meses)
monthly_wr['wr_rolling_3'] = monthly_wr['win_rate'].rolling(3, min_periods=1).mean()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Win Rate Mensal', 'Sinais por Mês'))

fig.add_trace(go.Bar(
    x=monthly_wr['ano_mes'], y=monthly_wr['win_rate'],
    name='Win Rate', marker_color=CYAN, opacity=0.6,
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=monthly_wr['ano_mes'], y=monthly_wr['wr_rolling_3'],
    name='Média 3 meses', line=dict(color=YELLOW, width=2),
), row=1, col=1)
fig.add_hline(y=signals_valid['hit'].mean()*100, line_dash='dash',
              line_color=GREEN, row=1, col=1,
              annotation_text=f"Global: {signals_valid['hit'].mean()*100:.1f}%")

fig.add_trace(go.Bar(
    x=monthly_wr['ano_mes'], y=monthly_wr['sinais'],
    name='Sinais', marker_color=MAGENTA, opacity=0.6,
), row=2, col=1)

fig.update_layout(height=700, title_text='Concept Drift: Evolução da Estratégia')
fig.show()

# Detectar tendência
from scipy.stats import linregress
x = np.arange(len(monthly_wr))
slope, intercept, r, p, se = linregress(x, monthly_wr['win_rate'])
print(f"\nTendência linear do win rate: slope = {slope:+.3f}%/mês (p = {p:.4f})")
if p < 0.05:
    print(f"Tendência significativa: win rate está {'subindo' if slope > 0 else 'caindo'}")
else:
    print("Sem tendência significativa ao longo do tempo")

## 4. Análise de Sequências Pré-Tragédia

In [ ]:
# Identificar tragédias e analisar o que acontece ANTES
tragedy_mask = df['low_streak'] >= TRAGEDY_STREAK
tragedy_starts = []

in_tragedy = False
for i, is_trag in enumerate(tragedy_mask):
    if is_trag and not in_tragedy:
        tragedy_starts.append(i)
        in_tragedy = True
    elif not is_trag:
        in_tragedy = False

print(f"Total de tragédias (12+ LOWs consecutivos): {len(tragedy_starts)}")

if len(tragedy_starts) > 0:
    # Coletar features 250 rounds antes de cada tragédia
    lookback = 250
    feature_cols = ['rolling_std_5', 'rolling_std_20', 'rolling_mean_20', 
                    'pct_low_20', 'pct_low_50', 'pct_low_100']
    available_cols = [c for c in feature_cols if c in df.columns]
    
    pre_tragedy_profiles = []
    for start in tragedy_starts:
        begin = max(0, start - lookback)
        window = df.iloc[begin:start][available_cols]
        profile = window.mean().to_dict()
        profile['tragedy_at'] = start
        profile['date'] = df.iloc[start]['date']
        pre_tragedy_profiles.append(profile)
    
    profiles_df = pd.DataFrame(pre_tragedy_profiles)
    
    # Comparar com valores normais
    normal_means = df[available_cols].mean()
    
    print(f"\nComparação Pré-Tragédia vs Normal:")
    print(f"{'Feature':>20} {'Normal':>10} {'Pré-Trag':>10} {'Diferença':>10}")
    print("-" * 55)
    for col in available_cols:
        normal_val = normal_means[col]
        pre_val = profiles_df[col].mean()
        diff = ((pre_val - normal_val) / normal_val) * 100
        marker = ' <<<' if abs(diff) > 10 else ''
        print(f"{col:>20} {normal_val:>10.4f} {pre_val:>10.4f} {diff:>+9.1f}%{marker}")

## 5. Resumo de Padrões Encontrados

In [ ]:
print("=" * 60)
print("RESUMO - PADRÕES ENCONTRADOS")
print("=" * 60)

global_wr = signals_valid['hit'].mean() * 100
total_signals = len(signals_valid)
total_days = (df['date'].max() - df['date'].min()).days

print(f"""
ESTRATÉGIA (trigger={trigger}, target={target}x):
  Win Rate Global: {global_wr:.1f}%
  Total de sinais: {total_signals:,} em {total_days:,} dias
  Sinais/dia: {total_signals/total_days:.1f}

HMM ({best_n} estados detectados):
  Regimes distintos identificados com diferentes perfis de volatilidade

PADRÕES TEMPORAIS:
  Existem variações significativas de win rate por hora e dia
  Melhores horários tendem a concentrar mais sinais positivos

CONCEPT DRIFT:
  Tendência: slope = {slope:+.3f}%/mês
  O comportamento do RNG {'muda' if p < 0.05 else 'se mantém estável'} ao longo do tempo

PRÉ-TRAGÉDIA:
  {len(tragedy_starts)} tragédias encontradas nos dados
  Sinais preditivos nas features de volatilidade pré-evento
""")